In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
# Install libraries
!pip install -q numpy scipy scikit-learn matplotlib torch torchvision gensim

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import CCA
from sklearn.metrics.pairwise import cosine_similarity
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from torchvision.datasets import CIFAR10
import gensim.downloader as api
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print("✓ Libraries imported!")

## Part 1: PCA Review

**Goal**: Find directions of maximum variance

In [ ]:
# Generate correlated 2D data
np.random.seed(42)
n_samples = 300

# Create data along a diagonal
X_original = np.random.randn(n_samples, 2)
X_original[:, 1] = X_original[:, 0] * 2 + np.random.randn(n_samples) * 0.5

print(f"Original data shape: {X_original.shape}")
print(f"Mean: {X_original.mean(axis=0)}")
print(f"Std: {X_original.std(axis=0)}")

In [ ]:
# Apply PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_original)

print(f"\nPCA Results:")
print(f"Explained variance ratio: {pca.explained_variance_ratio_}")
print(f"Principal components:\n{pca.components_}")
print(f"\nPC1 captures {pca.explained_variance_ratio_[0]*100:.1f}% variance")
print(f"PC2 captures {pca.explained_variance_ratio_[1]*100:.1f}% variance")

In [ ]:
# Visualize PCA
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Original data
ax1.scatter(X_original[:, 0], X_original[:, 1], alpha=0.5)
ax1.set_title('Original Data', fontsize=14)
ax1.set_xlabel('Feature 1')
ax1.set_ylabel('Feature 2')
ax1.grid(alpha=0.3)
ax1.axhline(0, color='k', linewidth=0.5)
ax1.axvline(0, color='k', linewidth=0.5)

# After PCA
ax2.scatter(X_pca[:, 0], X_pca[:, 1], alpha=0.5, color='orange')
ax2.set_title('After PCA (Rotated)', fontsize=14)
ax2.set_xlabel('PC1 (Max Variance)')
ax2.set_ylabel('PC2 (2nd Max Variance)')
ax2.grid(alpha=0.3)
ax2.axhline(0, color='k', linewidth=0.5)
ax2.axvline(0, color='k', linewidth=0.5)

plt.tight_layout()
plt.show()

print("Notice: PCA rotates data so variance is aligned with axes!")

## Part 2: Understanding CCA

**Goal**: Find projections that maximize correlation between two datasets

In [ ]:
# Generate correlated multimodal data
np.random.seed(42)
n = 200

# Latent variable (shared information)
z = np.random.randn(n, 1)

# Modality 1 (e.g., image features) - 3 dimensions
X = np.hstack([
    2 * z + np.random.randn(n, 1) * 0.5,  # Correlated with z
    np.random.randn(n, 1),                # Random
    z + np.random.randn(n, 1) * 0.3       # Weakly correlated
])

# Modality 2 (e.g., text features) - 3 dimensions
Y = np.hstack([
    -z + np.random.randn(n, 1) * 0.5,     # Negatively correlated with z
    np.random.randn(n, 1),                # Random
    -1.5 * z + np.random.randn(n, 1) * 0.4  # Strongly negatively correlated
])

print(f"Modality 1 (X) shape: {X.shape}")
print(f"Modality 2 (Y) shape: {Y.shape}")
print(f"\nBoth modalities share latent variable z")

In [ ]:
# Apply CCA
cca = CCA(n_components=2)
X_c, Y_c = cca.fit_transform(X, Y)

print("\nCCA Applied:")
print(f"X canonical variables shape: {X_c.shape}")
print(f"Y canonical variables shape: {Y_c.shape}")

In [ ]:
# Compute canonical correlations
corr_before_1 = np.corrcoef(X[:, 0], Y[:, 0])[0, 1]
corr_before_2 = np.corrcoef(X[:, 1], Y[:, 1])[0, 1]

corr_after_1 = np.corrcoef(X_c[:, 0], Y_c[:, 0])[0, 1]
corr_after_2 = np.corrcoef(X_c[:, 1], Y_c[:, 1])[0, 1]

print("\nCorrelations BEFORE CCA:")
print(f"  Dimension 1: {corr_before_1:.3f}")
print(f"  Dimension 2: {corr_before_2:.3f}")

print("\nCorrelations AFTER CCA:")
print(f"  Canonical 1: {corr_after_1:.3f} ✓ Maximized!")
print(f"  Canonical 2: {corr_after_2:.3f}")

print("\nCCA found linear combinations with highest correlation!")

In [ ]:
# Visualize CCA
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Before CCA
axes[0, 0].scatter(X[:, 0], Y[:, 0], alpha=0.5)
axes[0, 0].set_title(f'Before CCA: Dim 1\nCorr = {corr_before_1:.3f}')
axes[0, 0].set_xlabel('X[0]')
axes[0, 0].set_ylabel('Y[0]')
axes[0, 0].grid(alpha=0.3)

axes[0, 1].scatter(X[:, 1], Y[:, 1], alpha=0.5)
axes[0, 1].set_title(f'Before CCA: Dim 2\nCorr = {corr_before_2:.3f}')
axes[0, 1].set_xlabel('X[1]')
axes[0, 1].set_ylabel('Y[1]')
axes[0, 1].grid(alpha=0.3)

# After CCA
axes[1, 0].scatter(X_c[:, 0], Y_c[:, 0], alpha=0.5, color='orange')
axes[1, 0].set_title(f'After CCA: Canon 1\nCorr = {corr_after_1:.3f}')
axes[1, 0].set_xlabel('X_canonical[0]')
axes[1, 0].set_ylabel('Y_canonical[0]')
axes[1, 0].grid(alpha=0.3)

axes[1, 1].scatter(X_c[:, 1], Y_c[:, 1], alpha=0.5, color='orange')
axes[1, 1].set_title(f'After CCA: Canon 2\nCorr = {corr_after_2:.3f}')
axes[1, 1].set_xlabel('X_canonical[1]')
axes[1, 1].set_ylabel('Y_canonical[1]')
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Part 3: CCA for Image-Text Retrieval

In [ ]:
# Load CIFAR-10
transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

cifar = CIFAR10(root='./data', train=False, download=True, transform=transform)
classes = ['airplane', 'automobile', 'bird', 'cat', 'deer', 
           'dog', 'frog', 'horse', 'ship', 'truck']

print(f"CIFAR-10 loaded: {len(cifar)} images")

In [ ]:
# Load GloVe
print("Loading GloVe...")
glove = api.load('glove-wiki-gigaword-100')
print("✓ GloVe loaded")

In [ ]:
# Extract image features with ResNet
resnet = models.resnet50(pretrained=True).to(device)
resnet.eval()
feature_extractor = torch.nn.Sequential(*list(resnet.children())[:-1])

def extract_image_features(img_tensor):
    with torch.no_grad():
        img_tensor = img_tensor.unsqueeze(0).to(device)
        features = feature_extractor(img_tensor)
    return features.squeeze().cpu().numpy()

# Extract text features with GloVe
def extract_text_features(text):
    words = text.lower().split()
    word_vecs = [glove[w] for w in words if w in glove]
    if word_vecs:
        return np.mean(word_vecs, axis=0)
    return np.zeros(100)

print("✓ Feature extractors ready")

In [ ]:
# Extract features for subset
n_samples = 500
img_features = []
text_features = []
labels = []

print(f"Extracting features from {n_samples} samples...")
for i in range(n_samples):
    img, label = cifar[i]
    
    # Image features
    img_feat = extract_image_features(img)
    img_features.append(img_feat)
    
    # Text features
    text = classes[label]
    text_feat = extract_text_features(text)
    text_features.append(text_feat)
    
    labels.append(label)
    
    if (i+1) % 100 == 0:
        print(f"  Processed {i+1}/{n_samples}")

img_features = np.array(img_features)
text_features = np.array(text_features)
labels = np.array(labels)

print(f"\n✓ Features extracted")
print(f"  Image features: {img_features.shape}")
print(f"  Text features: {text_features.shape}")

In [ ]:
# Apply CCA
n_components = 50
cca_model = CCA(n_components=n_components)
img_cca, text_cca = cca_model.fit_transform(img_features, text_features)

print(f"✓ CCA applied with {n_components} components")
print(f"  Image CCA features: {img_cca.shape}")
print(f"  Text CCA features: {text_cca.shape}")

In [ ]:
# Cross-modal retrieval: Text → Image
def retrieve_images(query_text, top_k=5):
    # Extract query features
    query_feat = extract_text_features(query_text).reshape(1, -1)
    
    # Project to CCA space
    query_cca = cca_model.transform(np.zeros((1, img_features.shape[1])), query_feat)[1]
    
    # Compute similarities
    similarities = cosine_similarity(query_cca, img_cca)[0]
    top_indices = np.argsort(similarities)[::-1][:top_k]
    
    return top_indices, similarities[top_indices]

# Test retrieval
query = "cat"
indices, scores = retrieve_images(query, top_k=5)

print(f"\nQuery: '{query}'")
print(f"Top 5 results:")
for i, (idx, score) in enumerate(zip(indices, scores), 1):
    print(f"  {i}. {classes[labels[idx]]:12s} (similarity: {score:.3f})")

In [ ]:
# Visualize retrieval results
queries = ['dog', 'airplane', 'car']

for query in queries:
    indices, scores = retrieve_images(query, top_k=5)
    
    fig, axes = plt.subplots(1, 5, figsize=(15, 3))
    for i, (idx, score) in enumerate(zip(indices, scores)):
        img, _ = cifar[idx]
        # Denormalize
        img_display = img.permute(1, 2, 0).numpy()
        img_display = img_display * [0.229, 0.224, 0.225] + [0.485, 0.456, 0.406]
        img_display = np.clip(img_display, 0, 1)
        
        axes[i].imshow(img_display)
        axes[i].set_title(f"{classes[labels[idx]]}\n{score:.3f}", fontsize=9)
        axes[i].axis('off')
    
    plt.suptitle(f"Query: '{query}' → Retrieved Images (CCA)", fontsize=13)
    plt.tight_layout()
    plt.show()